<a href="https://colab.research.google.com/github/NataliiaFakas/TFG_BMW/blob/main/1_Montando_el_corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción de ventas de coches de BMW

# Diario TFG

---

# **Primer paso:** Limpieza de datasets

* Al final son todo datos de la UE porque es + sencillo encontrar los datos

# Filtrado ventas BMW por región (Europa)


---



In [ ]:
import pandas as pd

ruta = "/content/drive/MyDrive/TFG_BMW/datasets/BMW_dataset.csv"
df = pd.read_csv(ruta)

df.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low


Cargamos dataset y vamos a filtrar la región por Europa:

In [ ]:
df = df[df["Region"] == "Europe"]

In [ ]:
df["Region"].unique() #Comprobamos que ha funcionado
#Creamos un nuevo archivo con las regiones filtradas por Europa:
df.to_csv("/content/drive/MyDrive/TFG_BMW/datasets/BMW_dataset_europe.csv", index=False)

Ahora que tenemos BMW_dataset_europe.csv, vamos a filtrar la tasa de desempleo del dataset tasa_desempleo

# Filtrado dataset de la tasa de desempleo (UE)


---



In [ ]:
import pandas as pd

ruta = "/content/drive/MyDrive/TFG_BMW/datasets/tasa_desempleo_europe.csv"

# Leer CSV con separador ;
tasa_desempleo_europa = pd.read_csv(ruta, sep=';')

# Mostrar las primeras filas
tasa_desempleo_europa.head()


,Time,OBS_VALUE
0,2009,21.2
1,2010,22.4
2,2011,22.5
3,2012,24.4
4,2013,25.2


Ahora vamos a juntar la tasa de desempleo al dataset de BMW

***Apunte importante**: hay 3 rangos de edad, así que para cada año voy a sacar el promedio!!!!*

In [ ]:
import pandas as pd

# Cargamos el dataset de BMW filtrado por Europa
ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/BMW_dataset_europe.csv"
bmw_europa = pd.read_csv(ruta_bmw)

# Cargar dataset de desempleo (con sep=';')
ruta_desempleo = "/content/drive/MyDrive/TFG_BMW/datasets/tasa_desempleo_europe.csv"
tasa_desempleo_europa = pd.read_csv(ruta_desempleo, sep=';')

# Limpiamos espacios en los nombres de columnas (daba problemas)
tasa_desempleo_europa.columns = tasa_desempleo_europa.columns.str.strip()

# Renombramos columnas para unirlas luego
tasa_desempleo_europa = tasa_desempleo_europa.rename(columns={'Time':'Year', 'OBS_VALUE':'Unemployment_Rate'})

# Promediamos por año porque tenemos 3 rangos de edad en el dataset de Eurostat!!!!
tasa_desempleo_europa_avg = tasa_desempleo_europa.groupby('Year', as_index=False)['Unemployment_Rate'].mean()

# Unir datasets por 'Year'
bmw_europa = bmw_europa.merge(tasa_desempleo_europa_avg, on='Year', how='left')

# Mostrar primeras filas para comprobar ---
bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000


Ahora que lo hemos logrado, vamos a repetir el proceso con el crecimiento anual del pib en la UE
# Tasa de crecimiento anual del PIB en la UE (Banco Mundial)


---



Había empezado por usar un dataset del Banco Mundial, pero había muchas filas problemáticas con nombres que faltaban, caracteres extraños,...

Al final me quedé con el de Datosmacro.com

Vamos a cargarlo e imprimir el head de nuestro dataset:

In [ ]:
import pandas as pd

ruta_pib = "/content/drive/MyDrive/TFG_BMW/datasets/evolucion_PIB_zona_euro.csv"

pib = pd.read_csv(ruta_pib, sep=';')

Se ve bien, vamos a quitar los %, cambiar las comas por puntos y vamos a convertir las variaciones en floats

In [ ]:
# Quitar % y cambiar coma por punto
pib['Var. PIB (%)'] = pib['Var. PIB (%)'].str.replace('%', '')
pib['Var. PIB (%)'] = pib['Var. PIB (%)'].str.replace(',', '.')

# Convertir a float
pib['Var. PIB (%)'] = pib['Var. PIB (%)'].astype(float)

In [ ]:
pib.head()

,Fecha,Var. PIB (%)
0,2025,1.4
1,2024,0.9
2,2023,0.4
3,2022,3.6
4,2021,6.4


Ahora faltará renombrar las columnas para juntarlo al dataset completo:

In [ ]:
pib = pib.rename(columns={
    'Fecha': 'Year',
    'Var. PIB (%)': 'GDP_Growth'
}) #Esto para renombrar las columnas

pib['Year'] = pib['Year'].astype(int) #Vamos a pasar la columna Year a int por si acaso

In [ ]:
pib.head()

,Year,GDP_Growth
0,2025,1.4
1,2024,0.9
2,2023,0.4
3,2022,3.6
4,2021,6.4


Vemos que ya tiene el resultado que queremos, así que lo juntamos con el dataset conjunto:

In [ ]:
ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_europa_completo.csv" #Cargamos dataset
bmw_europa = pd.read_csv(ruta_bmw, sep=';') #quitamos las separaciones
bmw_europa = bmw_europa.replace(',', '.', regex=True) #cambiamos las comas por puntos
cols_numericas = [ #Aseguramos que son columnas numéricas, seguramente lo haya hecho antes pero bueno
    'Engine_Size_L',
    'Mileage_KM',
    'Price_USD',
    'Sales_Volume',
    'Unemployment_Rate'
]

for col in cols_numericas:
    bmw_europa[col] = pd.to_numeric(bmw_europa[col], errors='coerce')

bmw_europa['Year'] = bmw_europa['Year'].astype(int)

bmw_europa = bmw_europa.merge(pib, on='Year', how='left') #Juntamos ambos datasets, el del crecimiento del PIB junto con el total

bmw_europa.dtypes
bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4


Vemos que se ha unido correctamente, así que lo guardamos en la carpeta de Drive y seguimos con el siguiente indicador macroeconómico

In [ ]:
ruta_salida = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv" #Le pedí a ChatGPT que me lo hiciera y me ha generado uno nuevo en vez de usar el bmw_dataset_completo, pero bueno

ruta_salida = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"

bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4




---


# Key ECB Interest Rate: Deposit Facility

Para esta parte se han utilizado los datos del Banco Central Europeo para extraer el rate on the deposit facility.

* Concretamente se ha utilizado el Deposit Facility porque se utiliza como referencia para controlar la liquidez en la economía.
  * Concretamente es la tasa que los bancos reciben si dejan dinero "en depósito a un día" en el BCE durante un día sin prestarlo. Sirve para medir básicamente el coste de oportunidad de no prestar el dinero.

También estaban el Main Refinancing Operations pero esta es la tasa que los bancos pueden pedir prestado al BCE por una semana y el Marginal Lending Facility que es la tasa que los bancos pagan si necesitan dinero de manera urgente. Pero nos quedamos con el Deposit Facility porque es el más estable y el que parece más útil para añadir al dataset.

In [ ]:
ruta_deposit = "/content/drive/MyDrive/TFG_BMW/datasets/Key ECB interest rates Deposit Facility.csv"

deposit = pd.read_csv(ruta_deposit, sep=';')

deposit['Deposit facility'] = deposit['Deposit facility'].astype(str)  # asegurarnos de que sea string
deposit['Deposit facility'] = deposit['Deposit facility'].str.replace(r'[^0-9\.\-]', '', regex=True)

deposit['Deposit_facility'] = deposit['Deposit facility'].astype(float) # Ahora sí convertimos a float
deposit['Year'] = deposit['Year'].astype(int) # Asegurarnos de que 'Year' sea int

deposit = deposit[['Year', 'Deposit_facility']]

print(deposit.dtypes) # Comprobamos

deposit.head()

Year                  int64
Deposit_facility    float64
dtype: object


,Year,Deposit_facility
0,2025,2.00
1,2024,2.25
2,2023,2.50
3,2022,2.75
4,2021,3.00


Una vez hemos corregido el formato (había caracteres raros) y vemos que en efecto Year y Deposit_Facility son int y float respectivamente, pasamos a juntarlo con el dataset de BMW completo (el bmw_dataset_final)

In [ ]:
bmw_europa = bmw_europa.merge(deposit, on='Year', how='left')
bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25


Todo está bien así que vamos a guardarlo en Drive y pasamos al siguiente indicador macroeconómico

In [ ]:
ruta_salida = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"

bmw_europa.to_csv(ruta_salida, index=False)

bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25




---
# Variación de la inflación (IPC)
Vamos a continuar con la variación de la inflación que hemos sacado del BCE. Para ello, utilizamos el IPCA. Es armonizado para poder compararlo entre diferentes países.

Cabe a destacar que estaba por meses, así que cada año el IPC es el promedio de todos sus meses


In [ ]:
ruta_hicp = "/content/drive/MyDrive/TFG_BMW/datasets/HICP.csv"

hicp = pd.read_csv(ruta_hicp, sep=';')

hicp['Year'] = hicp['Year'].astype(int)
hicp['HICP'] = hicp['HICP'].str.replace(',', '.').astype(float)

# Comprobar tipos
print(hicp.dtypes)

hicp.head()

Year      int64
HICP    float64
dtype: object


,Year,HICP
0,2009,0.29
1,2010,1.62
2,2011,2.70
3,2012,2.50
4,2013,1.35


Una vez hecho esto, juntamos el IPCA con el dataset conjunto

In [ ]:
bmw_europa = bmw_europa.merge(hicp, on='Year', how='left')

ruta_salida = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"
bmw_europa.to_csv(ruta_salida, index=False)

bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP_x,HICP_y,HICP
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37,8.37,8.37
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19,1.19,1.19
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70,2.70,2.70
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46,5.46,5.46
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44,0.44,0.44


Todo está bien, guardamos los cambios

In [ ]:
bmw_europa.head() #se me ha duplicado el hicp, vamos a eliminar el hicp x y el y

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP_x,HICP_y,HICP
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37,8.37,8.37
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19,1.19,1.19
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70,2.70,2.70
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46,5.46,5.46
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44,0.44,0.44


In [ ]:
# Nos quedamos con una sola (por ejemplo HICP_y)
bmw_europa['HICP'] = bmw_europa['HICP_y']

# Eliminamos las duplicadas
bmw_europa.drop(columns=['HICP_x', 'HICP_y'], inplace=True)

ruta_salida = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"

bmw_europa.to_csv(ruta_salida, index=False)

bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44



---
# Crecimiento demográfico en la UE
No ha sido posible encontrar datos de la evolución del salario medio, así que he seguido con este indicador.

Vamos a cargarlo:



In [ ]:
import pandas as pd

ruta_demog = "/content/drive/MyDrive/TFG_BMW/datasets/Crecimiento_demog_europa.csv"

demog = pd.read_csv(
    ruta_demog,
    sep=";",
    encoding="latin1"
)
demog.head()

,Year,Austria,Belgium,Bulgaria,Switzerland,Cyprus,Czechia,Germany,Denmark,Estonia,...,Malta,Netherlands,Norway,Poland,Portugal,Romania,Sweden,Slovenia,Slovakia,PromedioEuropa
0,2010,8351643,10839905,7421766,7785806,819140,10462088,81802257,5534738,1333290,...,414027,16574989,4858199,38022869,10573479,20294683,9340682,2046976,5390410,"27978709,91"
1,2011,8375164,11000638,7369431,7870134,839751,10486731,80222065,5560628,1329660,...,414989,16655799,4920305,38062718,10572721,20199059,9415570,2050189,5392446,"27943664,5"
2,2012,8408121,11075889,7327224,7954662,862011,10505445,80327900,5580516,1325217,...,417546,16730348,4985870,38063792,10558950,20095996,9482855,2055496,5404322,"27995530,13"
3,2013,8451860,11137974,7202556,8039060,862854,10516125,80523746,5602628,1320174,...,421464,16779575,5051275,38062535,10503889,20020074,9555893,2058821,5410836,"28029028,13"
4,2014,8507786,11180840,7117453,8139631,860032,10512419,80767463,5627235,1315819,...,428156,16829289,5109056,38017856,10444092,19947311,9644864,2061085,5415949,"28067113,84"


Vamos a pasar las columnas a formato numérico

In [ ]:
# Primero aseguramos Año
demog["Year"] = pd.to_numeric(demog["Year"], errors="coerce")

# Convertir el resto de columnas a número
for col in demog.columns:
    if col != "Year":
        demog[col] = pd.to_numeric(
            demog[col].astype(str).str.replace(",", "."),
            errors="coerce"
        )

In [ ]:
demog.info()
demog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 34 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Year            16 non-null     int64  
 1   Austria         16 non-null     int64  
 2   Belgium         16 non-null     int64  
 3   Bulgaria        16 non-null     int64  
 4   Switzerland     16 non-null     int64  
 5   Cyprus          16 non-null     int64  
 6   Czechia         16 non-null     int64  
 7   Germany         16 non-null     int64  
 8   Denmark         16 non-null     int64  
 9   Estonia         16 non-null     int64  
 10  Greece          16 non-null     int64  
 11  Spain           16 non-null     int64  
 12  UE27            16 non-null     int64  
 13  Finland         16 non-null     int64  
 14  France          16 non-null     int64  
 15  Croatia         16 non-null     int64  
 16  Hungary         16 non-null     int64  
 17  Ireland         16 non-null     int64

,Year,Austria,Belgium,Bulgaria,Switzerland,Cyprus,Czechia,Germany,Denmark,Estonia,...,Malta,Netherlands,Norway,Poland,Portugal,Romania,Sweden,Slovenia,Slovakia,PromedioEuropa
0,2010,8351643,10839905,7421766,7785806,819140,10462088,81802257,5534738,1333290,...,414027,16574989,4858199,38022869,10573479,20294683,9340682,2046976,5390410,27978709.91
1,2011,8375164,11000638,7369431,7870134,839751,10486731,80222065,5560628,1329660,...,414989,16655799,4920305,38062718,10572721,20199059,9415570,2050189,5392446,27943664.50
2,2012,8408121,11075889,7327224,7954662,862011,10505445,80327900,5580516,1325217,...,417546,16730348,4985870,38063792,10558950,20095996,9482855,2055496,5404322,27995530.13
3,2013,8451860,11137974,7202556,8039060,862854,10516125,80523746,5602628,1320174,...,421464,16779575,5051275,38062535,10503889,20020074,9555893,2058821,5410836,28029028.13
4,2014,8507786,11180840,7117453,8139631,860032,10512419,80767463,5627235,1315819,...,428156,16829289,5109056,38017856,10444092,19947311,9644864,2061085,5415949,28067113.84


Ahora juntamos los datasets

In [ ]:
import pandas as pd

ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"

# Leemos todo como una sola columna
bmw_europa = pd.read_csv(ruta_bmw, sep=None, engine='python', encoding='latin1')

# Mostramos las primeras filas para comprobar
print(bmw_europa.head())
print(bmw_europa.columns)

  Model  Year  Region  Color Fuel_Type Transmission  Engine_Size_L  \
0    i8  2022  Europe  White    Diesel       Manual            1.8   
1    i8  2019  Europe  White  Electric       Manual            3.0   
2    M5  2011  Europe   Grey  Electric    Automatic            3.3   
3    i8  2023  Europe   Blue    Diesel    Automatic            3.8   
4    M3  2014  Europe    Red  Electric       Manual            2.5   

   Mileage_KM  Price_USD  Sales_Volume Sales_Classification  \
0      196741      55064          7949                 High   
1       35700      96257          4411                  Low   
2       78042      49507          9383                 High   
3       78573     118317          7168                 High   
4       74474      65464          9390                 High   

   Unemployment_Rate  GDP_Growth  Deposit_facility  HICP  
0           8.466667         3.6              2.75  8.37  
1           9.133333         1.6              3.50  1.19  
2          13.100000   

In [ ]:
bmw_europa.head()

,"Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP"
0,"i8,2022,Europe,White,Diesel,Manual,1.8,196741,..."
1,"i8,2019,Europe,White,Electric,Manual,3.0,35700..."
2,"M5,2011,Europe,Grey,Electric,Automatic,3.3,780..."
3,"i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573..."
4,"M3,2014,Europe,Red,Electric,Manual,2.5,74474,6..."


Se guardó mal el CSV:

In [ ]:
# Aseguramos que Year sea int
bmw_europa['Year'] = bmw_europa['Year'].astype(int)

# Creamos un diccionario Year → PromedioEuropa
promedio_map = dict(zip(demog['Year'], demog['PromedioEuropa']))

# Añadimos la columna PromedioEuropa
bmw_europa['PromedioEuropa'] = bmw_europa['Year'].map(promedio_map)

# Revisamos las primeras filas
bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP,PromedioEuropa
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37,28330546.03
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19,28330248.63
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70,27943664.50
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46,28441896.69
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44,28067113.84


In [ ]:
ruta_salida = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"
bmw_europa.to_csv(ruta_salida, sep=';', index=False, encoding='latin1')

In [ ]:
bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP,PromedioEuropa
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37,28330546.03
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19,28330248.63
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70,27943664.50
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46,28441896.69
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44,28067113.84




---
# Más indicadores
Vamos a añadir más indicadores para evitar problemas de overfitting y nuestro modelo pueda generalizar mejor.

Vamos a añadir los siguientes indicadores:
* Producción industrial
* Ventas minoristas (Retail Sales Index)
* Índice de confianza del consumidor
* Salario medio mensual


---
Vamos a cargar el dataset





In [6]:
import pandas as pd

ruta_indices = "/content/drive/MyDrive/TFG_BMW/datasets/producción industrial, ventas minoristas, confianza del consumidor, salario medio mensual.csv"

indices = pd.read_csv(
    ruta_indices,
    sep=';',
    decimal=',',
    encoding='latin1'
)

# Reemplazar ? por -
indices = indices.replace(r'\?', '-', regex=True)

# Convertir a numérico
for col in indices.columns:
    indices[col] = pd.to_numeric(indices[col], errors='coerce')

indices.head()

,Year,Evolución produccion industrial,Evolución ventas minoristas,Índice de confianza del consumidor,Evolución salario medio mensual
0,2010,8.0,0.02,-17,1.50
1,2011,NaN,0.01,-15,2.70
2,2012,NaN,0.00,-22,0.03
3,2013,NaN,0.01,-19,0.03
4,2014,NaN,0.01,-12,0.03


In [10]:
# la primera columna la lee entera como NaN, y no consigo que la lea bien
# así que voy a meterla manualmente
variaciones = {
    2010: 8,
    2011: 1,
    2012: -1,
    2013: 1,
    2014: 2,
    2015: 2,
    2016: 1,
    2017: 2,
    2018: 2,
    2019: -1,
    2020: -8,
    2021: 8,
    2022: 3,
    2023: -2,
    2024: -3
}
col = "Evolución produccion industrial"

indices[col] = indices.apply(
    lambda row: variaciones.get(row["Year"]) if pd.isna(row[col]) else row[col],
    axis=1
)


In [11]:
indices.head() #Ahora sí

,Year,Evolución produccion industrial,Evolución ventas minoristas,Índice de confianza del consumidor,Evolución salario medio mensual
0,2010,8.0,0.02,-17,1.50
1,2011,1.0,0.01,-15,2.70
2,2012,-1.0,0.00,-22,0.03
3,2013,1.0,0.01,-19,0.03
4,2014,2.0,0.01,-12,0.03


In [18]:
ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"
bmw_europa = pd.read_csv(ruta_bmw, sep=';')
bmw_europa = bmw_europa.replace(',', '.', regex=True)

bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP,PromedioEuropa
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37,28330546.03
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19,28330248.63
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70,27943664.50
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46,28441896.69
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44,28067113.84


In [19]:
bmw_europa = bmw_europa.merge(indices, on='Year', how='left') #Hacemos el merge

In [20]:
bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP,PromedioEuropa,Evolución produccion industrial,Evolución ventas minoristas,Índice de confianza del consumidor,Evolución salario medio mensual
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37,28330546.03,3.0,0.02,-24,0.06
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19,28330248.63,-1.0,0.01,-8,0.03
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70,27943664.50,1.0,0.01,-15,2.70
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46,28441896.69,-2.0,0.01,-18,0.04
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44,28067113.84,2.0,0.01,-12,0.03


In [22]:
# Aquí solamente vamos a renombrar columnas
bmw_europa = bmw_europa.rename(columns={
    "Evolución produccion industrial": "Prod_Ind",
    "Evolución ventas minoristas": "Ventas_Ret",
    "Índice de confianza del consumidor": "Conf_Cons",
    "Evolución salario medio mensual": "Salario_Mes",
    "PromedioEuropa": "Crec_demog"
})

# Guardamos el CSV
ruta_guardado = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"
bmw_europa.to_csv(ruta_guardado, sep=';', index=False, decimal='.')

bmw_europa.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Unemployment_Rate,GDP_Growth,Deposit_facility,HICP,Crec_demog,Prod_Ind,Ventas_Ret,Conf_Cons,Salario_Mes
0,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High,8.466667,3.6,2.75,8.37,28330546.03,3.0,0.02,-24,0.06
1,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low,9.133333,1.6,3.50,1.19,28330248.63,-1.0,0.01,-8,0.03
2,M5,2011,Europe,Grey,Electric,Automatic,3.3,78042,49507,9383,High,13.100000,1.7,2.00,2.70,27943664.50,1.0,0.01,-15,2.70
3,i8,2023,Europe,Blue,Diesel,Automatic,3.8,78573,118317,7168,High,8.366667,0.4,2.50,5.46,28441896.69,-2.0,0.01,-18,0.04
4,M3,2014,Europe,Red,Electric,Manual,2.5,74474,65464,9390,High,14.200000,1.4,3.25,0.44,28067113.84,2.0,0.01,-12,0.03




---
# Más variables
Vamos a añadir estas tres últimas variables
* Euríbor
* Evolución anual petróleo Brent Europa
* Evolución anual media EUR/USD

Vamos a cargar este dataset:


In [3]:
import pandas as pd

ruta_indices = "/content/drive/MyDrive/TFG_BMW/datasets/euríbor, precio del crudo, eur_usd.csv"

indices = pd.read_csv(
    ruta_indices,
    sep=';',
    decimal=',',
    encoding='latin1'
)

indices.head()

,Year,Euríbor,Precio del crudo,EUR/USD
0,2010,1.4,0.29,1.33
1,2011,1.7,0.40,1.39
2,2012,0.6,0.00,1.29
3,2013,0.4,-0.03,1.33
4,2014,0.2,-0.09,1.33


In [4]:
# Convertir todas las columnas a numérico
indices = indices.apply(pd.to_numeric, errors='coerce')

# Comprobar tipos
print(indices.dtypes)
indices.head()

Year                  int64
Euríbor             float64
Precio del crudo    float64
EUR/USD             float64
dtype: object


,Year,Euríbor,Precio del crudo,EUR/USD
0,2010,1.4,0.29,1.33
1,2011,1.7,0.40,1.39
2,2012,0.6,0.00,1.29
3,2013,0.4,-0.03,1.33
4,2014,0.2,-0.09,1.33


In [5]:
ruta_bmw_final = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"

bmw_final = pd.read_csv(ruta_bmw_final, sep=';')

bmw_final = bmw_final.replace(',', '.', regex=True)
bmw_final = bmw_final.apply(pd.to_numeric, errors='ignore')

/tmp/ipykernel_9836/1381379904.py:7: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  bmw_final = bmw_final.apply(pd.to_numeric, errors='ignore')


In [6]:
bmw_final["Year"] = pd.to_numeric(bmw_final["Year"], errors='coerce')
indices["Year"] = pd.to_numeric(indices["Year"], errors='coerce')

In [7]:
bmw_final = bmw_final.merge(indices, on='Year', how='left')
print(bmw_final.head())
print(bmw_final.isna().sum())
print(bmw_final.shape)

  Model  Year  Region  Color Fuel_Type Transmission  Engine_Size_L  \
0    i8  2022  Europe  White    Diesel       Manual            1.8   
1    i8  2019  Europe  White  Electric       Manual            3.0   
2    M5  2011  Europe   Grey  Electric    Automatic            3.3   
3    i8  2023  Europe   Blue    Diesel    Automatic            3.8   
4    M3  2014  Europe    Red  Electric       Manual            2.5   

   Mileage_KM  Price_USD  Sales_Volume  ... Deposit_facility  HICP  \
0      196741      55064          7949  ...             2.75  8.37   
1       35700      96257          4411  ...             3.50  1.19   
2       78042      49507          9383  ...             2.00  2.70   
3       78573     118317          7168  ...             2.50  5.46   
4       74474      65464          9390  ...             3.25  0.44   

    Crec_demog  Prod_Ind  Ventas_Ret  Conf_Cons  Salario_Mes  Euríbor  \
0  28330546.03       3.0        0.02        -24         0.06      2.5   
1  28330248.

In [8]:
ruta_guardado = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_final.csv"

bmw_final.to_csv(ruta_guardado, sep=';', index=False, decimal='.')

# Bibliografía
---
Ventas BMW (Kaggle)
* Dataset BMW:https:https://www.kaggle.com/datasets/y0ussefkandil/bmw-sales2010-2024?resource=download
  * Datos 2010-2024

* Notebooks de otras personas sobre este dataset:https://www.kaggle.com/datasets/y0ussefkandil/bmw-sales2010-2024/code
---
Unemployment by sex and age - annual data (eurostat)
* https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Unemployment_statistics

---
Evolución del PIB anual en la eurozona (Datos Macro)

* https://datosmacro.expansion.com/pib/zona-euro
---
Key ECB interest rates (Deposit facility)
* https://www.ecb.europa.eu/stats/policy_and_exchange_rates/key_ecb_interest_rates/html/index.es.html
---
Inflación (HICP Inflation rate - Total - Annual rate of change, Euro area, Monthly)
* https://www.ecb.europa.eu/stats/macroeconomic_and_sectoral/hicp/more/html/data.es.html
---
Population change - Demographic balance and crude rates at national level
* https://ec.europa.eu/eurostat/databrowser/view/demo_gind__custom_17386652/bookmark/table?lang=en&bookmarkId=1c4b8145-a6e5-48a5-afe4-e2b4e983c485&c=1752064755805
---
Evolución anual producción industrial – Eurozona (2010–2024, base 2021=100)
* https://db.nomics.world/Eurostat/teiis080
* https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Long_term_developments_in_industrial_production_-_results_from_short-term_statistics
* https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Industrial_production_%28volume%29_index_overview
* https://ec.europa.eu/eurostat/statistics-explained/index.php?title=National_accounts_and_GDP
* https://www.ceicdata.com/en/indicator/european-union/industrial-production-index-growth
* https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Industrial_production_(volume)_index_overview
---
Evolución anual ventas minoristas – Eurozona (2010–2024, base 2021=100)
* https://www.xtb.com/es/analisis-de-mercado/informe-de-ventas-minoristas-de-la-eurozona
* https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Towards_Digital_Decade_targets_for_Europe
* https://www.ine.es/prensa/icm_prensa.htm
* https://es.tradingeconomics.com/euro-area/retail-sales
